# Время выполнения и профилирование daslang

Перезапустите ядро после обновления. `%time` и `%%profile` выполняют код один раз и сохраняют изменения. `%timeit` повторяет код в независимых копиях исходного состояния. Файловые и сетевые эффекты при этом повторяются.

Время компиляции, создания контекста и восстановления состояния показано отдельно от выполнения.

In [1]:
var calls=0
def build_values(n:int):int {
    calls+=1
    var values:array<int>
    values |> reserve(n)
    var total=0
    for (i in range(n)) {
        values |> push(i)
        total+=i
    }
    return total
}

In [2]:
%time build_values(10000)

49995000

Compile: 3.904 ms
Simulate/relocate: 2.595 ms
Validate + restore state: 0.096 ms
Module initialization: 0.014 ms
Execute: 0.223 ms
Total session execution: 7.924 ms

In [3]:
calls

1

Следующая команда выполнит функцию шесть раз, но `calls` в основной сессии останется равным 1.

In [4]:
%timeit -r 3 -n 2 build_values(10000);

3 repeats x 2 isolated executions
Runtime: mean 0.222 ms; stddev 0.012 ms; best repeat 0.206 ms
Checkpoint setup (excluded): 6.995 ms/execution
Compile: 3.836 ms
Simulate/relocate: 2.287 ms
Validate + restore state: 0.062 ms
Module initialization: 0.011 ms
Execute: 0.222 ms
Total session execution: 7.281 ms
Notebook state unchanged; file/network/module lifecycle effects are not rolled back.

In [5]:
calls

1

In [6]:
%%time
var size=20000
build_values(size);

Compile: 3.881 ms
Simulate/relocate: 2.307 ms
Validate + restore state: 0.276 ms
Module initialization: 0.014 ms
Execute: 0.438 ms
Total session execution: 8.116 ms

Профилирование выполняет код один раз. Таблица показывает число вызовов, полное и собственное время, запрошенные байты памяти. Трассу можно открыть в Perfetto или chrome://tracing. Инструментирование замедляет выполнение.

In [7]:
%%profile --top 10 --trace "profiling-demo-trace.json"
build_values(size);

Compile: 3.817 ms
Simulate/relocate: 2.180 ms
Validate + restore state: 0.079 ms
Module initialization: 0.011 ms
Execute: 2.918 ms
Total session execution: 65.565 ms

Functions (instrumented wall time; inclusive includes callees):
calls  inclusive ms  self ms  allocated B  own B  function
    1         2.908    1.685        80000      0  build_values Ci
20000         1.222    1.222            0      0  builtin`push`4379756157886752001 X1<Y<numT>i>A C=Y<numT>i
    1         0.001    0.001        80000  80000  builtin`reserve`12130697888660093679 X1<Y<numT>i>A Ci
    1         2.909    0.001        80000      0  djnb_cell
Allocation bytes are requested heap/string bytes, including reallocation sizes; not live memory or peak usage.
Instrumentation adds overhead. Preparation, restoration and checkpoint serialization are not profiled.
Chrome/Perfetto trace: C:\Users\sanya\Documents\Codex\2026-10-11\jupyter\work\macro-inspection-pr\examples\profiling-demo-trace.json

In [8]:
calls

3

In [9]:
%%timeit -r 2
var temporary=123
build_values(5000);

2 repeats x 1 isolated executions
Runtime: mean 0.111 ms; stddev 0.008 ms; best repeat 0.102 ms
Checkpoint setup (excluded): 7.101 ms/execution
Compile: 3.804 ms
Simulate/relocate: 2.285 ms
Validate + restore state: 0.103 ms
Module initialization: 0.012 ms
Execute: 0.111 ms
Total session execution: 7.330 ms
Notebook state unchanged; file/network/module lifecycle effects are not rolled back.

In [10]:
calls

3

In [11]:
%reset